In [1]:
import pandas as pd
import numpy as np

# 1. Load the raw dataset
df = pd.read_csv('Coffee Shop Sales.csv')

In [4]:
df['transaction_date'] = pd.to_datetime(df['transaction_date'])
df['month'] = df['transaction_date'].dt.month
df['day_of_week'] = df['transaction_date'].dt.day_name()
df['is_weekend'] = df['transaction_date'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)
df['hour'] = pd.to_datetime(df['transaction_time'], format='%H:%M:%S').dt.hour
print("Done. Features [month, day_of_week, is_weekend, hour] created.")

Done. Features [month, day_of_week, is_weekend, hour] created.


In [6]:
def get_shift_info(hour):
    if 6 <= hour < 12: 
        return 'Morning', 1
    elif 12 <= hour < 17: 
        return 'Afternoon', 2
    else: 
        return 'Evening', 3

df[['shift', 'shift_num']] = df['hour'].apply(lambda x: pd.Series(get_shift_info(x)))
print("Done. Shifts assigned with numeric ID (1-Morning, 2-Afternoon, 3-Evening).")

Done. Shifts assigned with numeric ID (1-Morning, 2-Afternoon, 3-Evening).


In [8]:
demand_agg = df.groupby([
    'transaction_date', 
    'store_location', 
    'month', 
    'day_of_week', 
    'is_weekend', 
    'shift', 
    'shift_num'
])['transaction_qty'].sum().reset_index()

demand_agg = demand_agg.rename(columns={'transaction_qty': 'total_cups'})
print(f"Done. Compressed {len(df)} transactions into {len(demand_agg)} shift summaries.")

Done. Compressed 149116 transactions into 1623 shift summaries.


In [10]:
# Sort by Location -> Date -> Shift Number to ensure 'shift(1)' is truly the previous period
demand_agg = demand_agg.sort_values(['store_location', 'transaction_date', 'shift_num'])

# Calculate the cups sold in the actual previous shift
demand_agg['prev_cups'] = demand_agg.groupby('store_location')['total_cups'].shift(1).fillna(0)
print("Done. 'prev_cups' feature generated using chronological shift history.")

Done. 'prev_cups' feature generated using chronological shift history.


In [11]:
demand_agg.to_csv('Model1_Demand_Data.csv', index=False)
print("Done. File saved as 'Model1_Demand_Data.csv'.")

print("\n" + "="*40)
print("FEATURE ENGINEERING COMPLETE")
print("="*40)
print(demand_agg[['transaction_date', 'shift', 'total_cups', 'prev_cups']].head(10))

Done. File saved as 'Model1_Demand_Data.csv'.

FEATURE ENGINEERING COMPLETE
   transaction_date      shift  total_cups  prev_cups
2        2023-01-01    Morning          31        0.0
0        2023-01-01  Afternoon         156       31.0
1        2023-01-01    Evening          90      156.0
11       2023-01-02    Morning          38       90.0
9        2023-01-02  Afternoon         163       38.0
10       2023-01-02    Evening         100      163.0
20       2023-01-03    Morning          35      100.0
18       2023-01-03  Afternoon         166       35.0
19       2023-01-03    Evening          86      166.0
29       2023-01-04    Morning          23       86.0
